# 11a GFD Boundary Matching

This notebook matches Global Flood Database geoBoundaries ADM2 export records to the verified DFSI district table. It is deliberately conservative: accepted matches must be exact or manually reviewed. Fuzzy matching is used only to suggest candidates for review and never to merge records automatically.

If Earth Engine exports are not yet available under `data/processed/gfd_external_validation/`, the notebook writes the required audit outputs and a manual crosswalk template, then marks the pilot as blocked by missing exports.

In [1]:
from pathlib import Path
import re
import hashlib
import difflib
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
BASELINE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dfsi_verified_baseline.csv'
GFD_DIR = PROJECT_ROOT / 'data' / 'processed' / 'gfd_external_validation'
GFD_METRICS_PATH = GFD_DIR / 'gfd_india_adm2_metrics.csv'
GFD_BOUNDARIES_PATH = GFD_DIR / 'gfd_india_adm2_boundaries.csv'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
MANUAL_DIR = PROJECT_ROOT / 'data' / 'manual'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_DIR.mkdir(parents=True, exist_ok=True)
GFD_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
RAW_HASHES_BEFORE = {p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}

baseline = pd.read_csv(BASELINE_PATH)
complete = baseline.query("completeness_status == 'complete' and join_status == 'one_to_one_comparison_record'").copy()
print('Verified DFSI baseline rows:', len(baseline))
print('Complete reconstruction rows:', len(complete))
assert 700 <= len(baseline) <= 730
assert 700 <= len(complete) <= 715

Verified DFSI baseline rows: 720
Complete reconstruction rows: 709


## Source Columns

The DFSI source is the frozen verified baseline. Column choices are read directly from the file and documented in `outputs/tables/gfd_input_file_audit.csv`.

In [2]:
selected_columns = {
    'original_district_name': 'district_name',
    'standardised_district_name': 'available_district_review_key',
    'state_name': 'State_Name',
    'standardised_state_name': 'available_state_review_key',
    'published_dfsi': 'published_dfsi',
    'reconstructed_dfsi': 'reconstructed_dfsi',
    'published_rank': 'published_rank',
    'reconstructed_rank': 'reconstructed_rank',
    'component_flood_events': 'flood_event_count',
    'component_duration': 'mean_flood_duration',
    'component_fatalities': 'fatalities',
    'component_injured': 'injured',
    'component_flooded_area': 'corrected_percent_flooded_area',
    'component_population': 'population',
    'join_status': 'join_status',
    'completeness_status': 'completeness_status',
}
missing = [col for col in selected_columns.values() if col not in baseline.columns]
assert not missing, f'Missing expected DFSI columns: {missing}'
display(pd.DataFrame([{'role':k, 'column':v} for k,v in selected_columns.items()]))

,role,column
0,original_district_name,district_name
1,standardised_district_name,available_district_review_key
2,state_name,State_Name
3,standardised_state_name,available_state_review_key
4,published_dfsi,published_dfsi
5,reconstructed_dfsi,reconstructed_dfsi
6,published_rank,published_rank
7,reconstructed_rank,reconstructed_rank
8,component_flood_events,flood_event_count
9,component_duration,mean_flood_duration


## Load Boundary Export or Mark Missing

In [3]:
if GFD_METRICS_PATH.exists():
    boundaries = pd.read_csv(GFD_METRICS_PATH)
    boundary_source_path = GFD_METRICS_PATH
    boundary_available = True
elif GFD_BOUNDARIES_PATH.exists():
    boundaries = pd.read_csv(GFD_BOUNDARIES_PATH)
    boundary_source_path = GFD_BOUNDARIES_PATH
    boundary_available = True
else:
    boundaries = pd.DataFrame(columns=['boundary_shape_id','boundary_district_name'])
    boundary_source_path = GFD_METRICS_PATH
    boundary_available = False

print('GFD boundary source exists:', boundary_available)
print('Boundary source path:', boundary_source_path.relative_to(PROJECT_ROOT))
print('Boundary rows:', len(boundaries))
print('Boundary columns:', boundaries.columns.tolist())

boundary_name_col = 'boundary_district_name' if 'boundary_district_name' in boundaries.columns else None
boundary_id_col = 'boundary_shape_id' if 'boundary_shape_id' in boundaries.columns else None
boundary_state_col = None
for candidate in ['boundary_state_name', 'state_name', 'shapeState', 'shapeName_1', 'ADM1_NAME']:
    if candidate in boundaries.columns:
        boundary_state_col = candidate
        break
print('Boundary district column:', boundary_name_col)
print('Boundary state column:', boundary_state_col)


GFD boundary source exists: True
Boundary source path: data/processed/gfd_external_validation/gfd_india_adm2_metrics.csv
Boundary rows: 735
Boundary columns: ['system:index', 'boundary_district_name', 'boundary_shape_group', 'boundary_shape_id', 'boundary_shape_type', 'district_area_km2', '.geo', 'gfd_observed_event_count', 'cumulative_flooded_area_event_km2', 'cumulative_inundation_area_days_km2_days', 'ever_flooded_area_km2', 'first_observed_event_date', 'last_observed_event_date', 'gfd_event_collection_count', 'ever_flooded_fraction_pct', 'cumulative_flooded_area_event_fraction', 'cumulative_inundation_area_days_per_km2', 'mean_flooded_pixel_event_count', 'maximum_flooded_pixel_event_count', 'mean_clear_perc_over_observed_flood_pixels']
Boundary district column: boundary_district_name
Boundary state column: None


## Matching Helpers

Normalisation handles case, whitespace, punctuation and a small transparent set of spelling/renaming variants. District splits are not merged automatically.

In [4]:
COMMON_VARIANTS = {
    'bangalore': 'bengaluru',
    'bangalore rural': 'bengaluru rural',
    'bangalore urban': 'bengaluru urban',
    'belgaum': 'belagavi',
    'bellary': 'ballari',
    'bid': 'beed',
    'bijapur': 'vijayapura',
    'gulbarga': 'kalaburagi',
    'mysore': 'mysuru',
    'tumkur': 'tumakuru',
    'cuddapah': 'ysr',
    'puducherry': 'pondicherry',
    'gurgaon': 'gurugram',
    'mewat': 'nuh',
    'ahmadabad': 'ahmedabad',
    'medinipur west': 'paschim medinipur',
    'medinipur east': 'purba medinipur',
    'north twenty four pargan': 'north 24 parganas',
    'south twenty four pargan': 'south 24 parganas',
}

def norm_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = COMMON_VARIANTS.get(text, text)
    return text

def make_key(state, district, use_state=True):
    d = norm_text(district)
    if pd.isna(d):
        return pd.NA
    if use_state:
        s = norm_text(state)
        if not pd.isna(s):
            return f'{s}||{d}'
    return d

match_scope_note = 'Boundary state was available; exact/manual matching uses state + district.' if boundary_state_col else 'Boundary state was not available in the GFD export; matching is district-name based and duplicate/ambiguous names are not auto-merged.'
print(match_scope_note)

Boundary state was not available in the GFD export; matching is district-name based and duplicate/ambiguous names are not auto-merged.


## Manual Crosswalk Template

In [5]:
crosswalk_path = MANUAL_DIR / 'gfd_district_crosswalk_template.csv'
crosswalk_columns = [
    'boundary_shape_id','boundary_district_name','boundary_state_name',
    'dfsi_district_name','dfsi_state_name','dfsi_record_key',
    'match_method','review_status','review_notes'
]
if not crosswalk_path.exists():
    pd.DataFrame(columns=crosswalk_columns).to_csv(crosswalk_path, index=False)
manual_crosswalk = pd.read_csv(crosswalk_path)
for col in crosswalk_columns:
    if col not in manual_crosswalk.columns:
        manual_crosswalk[col] = pd.NA
print('Manual crosswalk template:', crosswalk_path.relative_to(PROJECT_ROOT), 'rows:', len(manual_crosswalk))

Manual crosswalk template: data/manual/gfd_district_crosswalk_template.csv rows: 0


## Exact and Manual Matching

In [6]:
dfsi = baseline.copy()
dfsi['dfsi_record_key'] = dfsi.index.astype(str)
dfsi['dfsi_original_key'] = dfsi.apply(lambda r: make_key(r['State_Name'], r['district_name'], use_state=bool(boundary_state_col)), axis=1)
dfsi['dfsi_standardised_key'] = dfsi.apply(lambda r: make_key(r['State_Name'], r['available_district_review_key'], use_state=bool(boundary_state_col)), axis=1)

if boundary_available and boundary_name_col:
    b = boundaries.copy()
    b['boundary_record_key'] = b.index.astype(str)
    b['boundary_state_for_match'] = b[boundary_state_col] if boundary_state_col else pd.NA
    b['boundary_original_key'] = b.apply(lambda r: make_key(r['boundary_state_for_match'], r[boundary_name_col], use_state=bool(boundary_state_col)), axis=1)
else:
    b = pd.DataFrame(columns=list(boundaries.columns) + ['boundary_record_key','boundary_state_for_match','boundary_original_key'])

matches=[]
ambiguous=[]

def add_exact_matches(boundary_frame, dfsi_key_col, method):
    if boundary_frame.empty:
        return
    left_counts = boundary_frame['boundary_original_key'].value_counts(dropna=True)
    right_counts = dfsi[dfsi_key_col].value_counts(dropna=True)
    candidate = boundary_frame.merge(
        dfsi,
        left_on='boundary_original_key',
        right_on=dfsi_key_col,
        how='inner',
        suffixes=('_boundary','_dfsi')
    )
    for _, row in candidate.iterrows():
        key = row['boundary_original_key']
        if left_counts.get(key, 0) == 1 and right_counts.get(key, 0) == 1:
            matches.append({
                'boundary_record_key': row['boundary_record_key'],
                'boundary_shape_id': row.get(boundary_id_col, pd.NA) if boundary_id_col else pd.NA,
                'boundary_district_name': row.get(boundary_name_col, pd.NA) if boundary_name_col else pd.NA,
                'boundary_state_name': row.get('boundary_state_for_match', pd.NA),
                'dfsi_record_key': row['dfsi_record_key'],
                'dfsi_district_name': row['district_name'],
                'dfsi_state_name': row['State_Name'],
                'match_key': key,
                'match_method': method,
                'match_status': 'accepted_unique',
                'notes': match_scope_note,
            })
        else:
            ambiguous.append({
                'match_key': key,
                'boundary_record_key': row['boundary_record_key'],
                'boundary_shape_id': row.get(boundary_id_col, pd.NA) if boundary_id_col else pd.NA,
                'boundary_district_name': row.get(boundary_name_col, pd.NA) if boundary_name_col else pd.NA,
                'dfsi_record_key': row['dfsi_record_key'],
                'dfsi_district_name': row['district_name'],
                'dfsi_state_name': row['State_Name'],
                'candidate_match_method': method,
                'boundary_key_count': int(left_counts.get(key, 0)),
                'dfsi_key_count': int(right_counts.get(key, 0)),
                'ambiguity_reason': 'non_unique_boundary_or_dfsi_key',
            })

add_exact_matches(b, 'dfsi_original_key', 'exact_original')
matched_boundary = {m['boundary_record_key'] for m in matches}
remaining_b = b.loc[~b['boundary_record_key'].isin(matched_boundary)].copy() if not b.empty else b.copy()
add_exact_matches(remaining_b, 'dfsi_standardised_key', 'exact_standardised')

# Manual crosswalk accepts only rows explicitly marked accepted and complete.
manual_accepted = manual_crosswalk.loc[manual_crosswalk['review_status'].astype(str).str.lower().eq('accepted')].copy()
for _, row in manual_accepted.iterrows():
    matches.append({
        'boundary_record_key': pd.NA,
        'boundary_shape_id': row.get('boundary_shape_id', pd.NA),
        'boundary_district_name': row.get('boundary_district_name', pd.NA),
        'boundary_state_name': row.get('boundary_state_name', pd.NA),
        'dfsi_record_key': row.get('dfsi_record_key', pd.NA),
        'dfsi_district_name': row.get('dfsi_district_name', pd.NA),
        'dfsi_state_name': row.get('dfsi_state_name', pd.NA),
        'match_key': pd.NA,
        'match_method': 'manual_crosswalk',
        'match_status': 'accepted_unique',
        'notes': row.get('review_notes', 'Manual crosswalk accepted by reviewer.'),
    })

unique_matches = pd.DataFrame(matches)
ambiguous_matches = pd.DataFrame(ambiguous)
if not unique_matches.empty:
    duplicate_boundary = unique_matches['boundary_shape_id'].notna() & unique_matches['boundary_shape_id'].duplicated(keep=False)
    duplicate_dfsi = unique_matches['dfsi_record_key'].notna() & unique_matches['dfsi_record_key'].duplicated(keep=False)
    ambiguous_from_matches = unique_matches.loc[duplicate_boundary | duplicate_dfsi].copy()
    if not ambiguous_from_matches.empty:
        ambiguous_from_matches['ambiguity_reason'] = 'manual_or_exact_match_not_unique_after_combination'
        ambiguous_matches = pd.concat([ambiguous_matches, ambiguous_from_matches], ignore_index=True)
        unique_matches = unique_matches.loc[~(duplicate_boundary | duplicate_dfsi)].copy()

print('Accepted unique matches:', len(unique_matches))
print('Ambiguous candidate matches:', len(ambiguous_matches))

Accepted unique matches: 668
Ambiguous candidate matches: 8


## Fuzzy Suggestions for Review Only

Fuzzy suggestions are written as notes in unmatched audit tables. They are never accepted automatically.

In [7]:
accepted_dfsi_keys = set(unique_matches['dfsi_record_key'].astype(str)) if not unique_matches.empty else set()
accepted_boundary_ids = set(unique_matches['boundary_shape_id'].dropna().astype(str)) if not unique_matches.empty else set()

unmatched_dfsi = dfsi.loc[~dfsi['dfsi_record_key'].astype(str).isin(accepted_dfsi_keys)].copy()
if boundary_available and not b.empty:
    boundary_id_series = b[boundary_id_col].astype(str) if boundary_id_col else b['boundary_record_key'].astype(str)
    unmatched_boundary = b.loc[~boundary_id_series.isin(accepted_boundary_ids)].copy()
else:
    unmatched_boundary = b.copy()

boundary_norm_names = b[boundary_name_col].dropna().map(norm_text).dropna().unique().tolist() if boundary_available and boundary_name_col else []
dfsi_norm_names = dfsi['district_name'].dropna().map(norm_text).dropna().unique().tolist()

def suggestions(name, candidates):
    n = norm_text(name)
    if pd.isna(n) or not candidates:
        return ''
    return '; '.join(difflib.get_close_matches(n, candidates, n=5, cutoff=0.82))

unmatched_dfsi['fuzzy_boundary_suggestions_for_review_only'] = unmatched_dfsi['district_name'].map(lambda x: suggestions(x, boundary_norm_names))
if not unmatched_boundary.empty and boundary_name_col:
    unmatched_boundary['fuzzy_dfsi_suggestions_for_review_only'] = unmatched_boundary[boundary_name_col].map(lambda x: suggestions(x, dfsi_norm_names))
else:
    unmatched_boundary['fuzzy_dfsi_suggestions_for_review_only'] = pd.Series(dtype='object')

# If the export is missing, retain all DFSI rows as unmatched with an explicit blocker note.
if not boundary_available:
    unmatched_dfsi['unmatched_reason'] = 'gfd_adm2_metrics_export_missing'
else:
    unmatched_dfsi['unmatched_reason'] = 'no_unique_accepted_boundary_match'
unmatched_boundary['unmatched_reason'] = 'no_unique_accepted_dfsi_match' if boundary_available else 'gfd_adm2_metrics_export_missing'

boundary_note = pd.DataFrame([{
    'note_type': 'boundary_vintage_limitation',
    'note': 'geoBoundaries ADM2 boundary vintage may not match the administrative vintage used by the DFSI source tables. District splits or renamings are not merged automatically and must be reviewed manually.'
}])

## Save Matching Outputs

In [8]:
match_audit = pd.DataFrame([{
    'audit_item': 'dfsi_source_file',
    'value': str(BASELINE_PATH.relative_to(PROJECT_ROOT)),
    'notes': 'Verified district-level DFSI baseline.'
},{
    'audit_item': 'gfd_boundary_source_file',
    'value': str(boundary_source_path.relative_to(PROJECT_ROOT)),
    'notes': 'Exists' if boundary_available else 'Missing; run GEE boundary or metrics export first.'
},{
    'audit_item': 'dfsi_records_total',
    'value': len(baseline),
    'notes': 'One-to-one baseline records.'
},{
    'audit_item': 'dfsi_complete_records',
    'value': len(complete),
    'notes': 'Complete reconstruction records.'
},{
    'audit_item': 'boundary_records_total',
    'value': len(boundaries),
    'notes': 'Rows in GFD ADM2 metrics or boundary export.'
},{
    'audit_item': 'accepted_unique_matches',
    'value': len(unique_matches),
    'notes': 'Exact/manual only; fuzzy suggestions excluded.'
},{
    'audit_item': 'ambiguous_matches',
    'value': len(ambiguous_matches),
    'notes': 'Preserved for review.'
},{
    'audit_item': 'unmatched_dfsi_records',
    'value': len(unmatched_dfsi),
    'notes': 'Includes all records if GFD export is missing.'
},{
    'audit_item': 'unmatched_boundary_records',
    'value': len(unmatched_boundary),
    'notes': 'Boundary records without accepted DFSI match.'
},{
    'audit_item': 'matching_scope_note',
    'value': match_scope_note,
    'notes': 'State used only if available in boundary export.'
}])

# Ensure stable columns for empty outputs.
unique_cols = ['boundary_record_key','boundary_shape_id','boundary_district_name','boundary_state_name','dfsi_record_key','dfsi_district_name','dfsi_state_name','match_key','match_method','match_status','notes']
ambiguous_cols = ['match_key','boundary_record_key','boundary_shape_id','boundary_district_name','dfsi_record_key','dfsi_district_name','dfsi_state_name','candidate_match_method','boundary_key_count','dfsi_key_count','ambiguity_reason']
for col in unique_cols:
    if col not in unique_matches.columns: unique_matches[col] = pd.Series(dtype='object')
for col in ambiguous_cols:
    if col not in ambiguous_matches.columns: ambiguous_matches[col] = pd.Series(dtype='object')

match_audit.to_csv(TABLE_DIR / 'gfd_boundary_match_audit.csv', index=False)
unique_matches[unique_cols].to_csv(TABLE_DIR / 'gfd_unique_matches.csv', index=False)
unmatched_dfsi.to_csv(TABLE_DIR / 'gfd_unmatched_dfsi_records.csv', index=False)
unmatched_boundary.to_csv(TABLE_DIR / 'gfd_unmatched_boundary_records.csv', index=False)
ambiguous_matches[ambiguous_cols].to_csv(TABLE_DIR / 'gfd_ambiguous_matches.csv', index=False)

# Append the boundary-vintage note to match audit as a separate small companion in the same audit table format.
existing = pd.read_csv(TABLE_DIR / 'gfd_boundary_match_audit.csv')
existing = pd.concat([existing, pd.DataFrame([{'audit_item':'boundary_vintage_limitation','value':boundary_note.iloc[0]['note'],'notes':'Limitation for interpretation.'}])], ignore_index=True)
existing.to_csv(TABLE_DIR / 'gfd_boundary_match_audit.csv', index=False)

for p in ['gfd_boundary_match_audit.csv','gfd_unique_matches.csv','gfd_unmatched_dfsi_records.csv','gfd_unmatched_boundary_records.csv','gfd_ambiguous_matches.csv']:
    path = TABLE_DIR / p
    assert path.exists()
print('Saved matching outputs to outputs/tables/.')
RAW_HASHES_AFTER = {p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
assert RAW_HASHES_BEFORE == RAW_HASHES_AFTER, 'data/raw changed during boundary matching.'
display(existing)

Saved matching outputs to outputs/tables/.


,audit_item,value,notes
0,dfsi_source_file,data/processed/dfsi_verified_baseline.csv,Verified district-level DFSI baseline.
1,gfd_boundary_source_file,data/processed/gfd_external_validation/gfd_ind...,Exists
2,dfsi_records_total,720,One-to-one baseline records.
3,dfsi_complete_records,709,Complete reconstruction records.
4,boundary_records_total,735,Rows in GFD ADM2 metrics or boundary export.
5,accepted_unique_matches,668,Exact/manual only; fuzzy suggestions excluded.
6,ambiguous_matches,8,Preserved for review.
7,unmatched_dfsi_records,52,Includes all records if GFD export is missing.
8,unmatched_boundary_records,67,Boundary records without accepted DFSI match.
9,matching_scope_note,Boundary state was not available in the GFD ex...,State used only if available in boundary export.
